# 04 - Validacion cruzada

Evalua la capacidad de generalizacion del MLP usando Stratified K-Fold.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.config import BATCH_SIZE, DATASET_DIR, METRICS_DIR, EPOCHS, RANDOM_STATE
from src.preprocessing import load_split
from src.model import PureTensorFlowMLP, predict, train_mlp
from src.evaluation import compute_metrics

I0000 00:00:1781938386.675942    1810 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781938386.716101    1810 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781938387.718503    1810 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)
x, y, _ = load_split(DATASET_DIR, 'train')
x.shape, y.shape

((148, 4096), (148,))

In [3]:
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
rows = []

for fold, (train_idx, val_idx) in enumerate(skf.split(x, y), start=1):
    model = PureTensorFlowMLP(input_dim=x.shape[1], seed=RANDOM_STATE + fold)
    train_mlp(model, x[train_idx], y[train_idx], x_val=x[val_idx], y_val=y[val_idx], epochs=EPOCHS, batch_size=BATCH_SIZE)
    y_prob = predict(model, x[val_idx])
    rows.append({'fold': fold, **compute_metrics(y[val_idx], y_prob)})

cv_results = pd.DataFrame(rows)
cv_results.to_csv(METRICS_DIR / 'cross_validation_results.csv', index=False)
cv_results

I0000 00:00:1781938398.471208    1810 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1763 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Epoca 1/30 - loss: 1.3071 - accuracy: 0.5270 - val_loss: 0.6728 - val_accuracy: 0.5667
Epoca 2/30 - loss: 0.6957 - accuracy: 0.6499 - val_loss: 0.6560 - val_accuracy: 0.6000
Epoca 3/30 - loss: 0.5794 - accuracy: 0.7486 - val_loss: 0.5286 - val_accuracy: 0.7333
Epoca 4/30 - loss: 0.4845 - accuracy: 0.8480 - val_loss: 0.4808 - val_accuracy: 0.7000
Epoca 5/30 - loss: 0.4375 - accuracy: 0.8061 - val_loss: 0.5109 - val_accuracy: 0.7000
Epoca 6/30 - loss: 0.4404 - accuracy: 0.8374 - val_loss: 0.3966 - val_accuracy: 0.8667
Epoca 7/30 - loss: 0.3515 - accuracy: 0.8487 - val_loss: 0.4123 - val_accuracy: 0.7333
Epoca 8/30 - loss: 0.3318 - accuracy: 0.8530 - val_loss: 0.3541 - val_accuracy: 0.8667
Epoca 9/30 - loss: 0.2735 - accuracy: 0.9183 - val_loss: 0.3383 - val_accuracy: 0.8667
Epoca 10/30 - loss: 0.2402 - accuracy: 0.9382 - val_loss: 0.3213 - val_accuracy: 0.8667
Epoca 11/30 - loss: 0.2219 - accuracy: 0.9382 - val_loss: 0.3662 - val_accuracy: 0.8667
Epoca 12/30 - loss: 0.1980 - accuracy: 0.

,fold,accuracy,precision,recall,f1_score
0,1,0.900000,0.812500,1.000000,0.896552
1,2,0.866667,0.812500,0.928571,0.866667
2,3,0.933333,0.875000,1.000000,0.933333
3,4,0.862069,0.909091,0.769231,0.833333
4,5,0.896552,0.857143,0.923077,0.888889


In [4]:
summary = cv_results.drop(columns=['fold']).agg(['mean', 'std'])
summary.to_csv(METRICS_DIR / 'cross_validation_summary.csv')
summary

,accuracy,precision,recall,f1_score
mean,0.891724,0.853247,0.924176,0.883755
std,0.028855,0.041616,0.094243,0.037022
